# Visualization 2

In [11]:
import pandas as pd
import altair as alt

alt.data_transformers.enable("json")   # let Altair handle the large dataset (lifts the 5000-row limit)

DataTransformerRegistry.enable('json')

In [12]:
data_dir="../data/"

In [13]:
names = pd.read_csv(f"{data_dir}dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2372651,2,EMMA,1905,37,4
1986543,2,AURIANE,2002,76,9
3425104,2,RACHEL,2006,49,7
2292624,2,EDMONDE,1910,75,8
3285515,2,NADINE,1975,03,9


Let's explore the data

In [14]:
print("Number of distinct names :", names.preusuel.nunique())
print("Number of departments    :", names.dpt.nunique())
print(f"Years from {names.annais.min()}  to  {names.annais.max()} ")

Number of distinct names : 15270
Number of departments    : 99
Years from 1900  to  2020 


Let's display the most popular names in France

In [15]:
popularity = names.groupby("preusuel")["nombre"].sum()
popularity.sort_values(ascending=False).head(10)

preusuel
MARIE       2256072
JEAN        1913130
PIERRE       891794
MICHEL       818025
ANDRÉ        709633
JEANNE       556903
PHILIPPE     535355
LOUIS        523576
RENÉ         514560
ALAIN        504106
Name: nombre, dtype: int64

Let's display the most popular names per departement

In [16]:
popularity_dpt    = names.groupby(["preusuel", "dpt"])["nombre"].sum()
popularity_dpt.sort_values(ascending=False).head(10)

preusuel  dpt
MARIE     974    202430
JEAN      75     148391
MARIE     29     100282
PIERRE    75      84480
MICHEL    75      83471
JACQUES   75      77569
MARIE     59      75294
          75      73518
JEAN      59      72261
ANDRÉ     75      71254
Name: nombre, dtype: int64

Let's add a column that shows the exact decade of birth.

In [17]:
names["annais"] = pd.to_numeric(names["annais"], errors="coerce")
names.dropna(subset=["annais"], inplace=True)
names["annais"] = names["annais"].astype(int)

names["decade"] = (names["annais"] // 10) * 10
names.head()

,sexe,preusuel,annais,dpt,nombre,decade
10885,1,AADIL,1983,84,3,1980
10886,1,AADIL,1992,92,3,1990
10888,1,AAHIL,2016,95,3,2010
10892,1,AARON,1962,75,3,1960
10893,1,AARON,1976,75,3,1970


In [18]:
# department shares per decade and  name
birth_numbers_per_decade_per_dpt  = names.groupby(["decade", "preusuel", "dpt"], as_index=False)["nombre"].sum()
total_per_decade = birth_numbers_per_decade_per_dpt.groupby(["decade", "preusuel"])["nombre"].transform("sum")
birth_numbers_per_decade_per_dpt["share"] = birth_numbers_per_decade_per_dpt["nombre"] / total_per_decade
birth_numbers_per_decade_per_dpt.head()

,decade,preusuel,dpt,nombre,share
0,1900,ABDON,59,22,0.333333
1,1900,ABDON,66,31,0.469697
2,1900,ABDON,972,13,0.196970
3,1900,ABEL,01,6,0.001277
4,1900,ABEL,02,95,0.020213


In [19]:
# normalized HHI (Herfindahl index) per decade and name to measure the concentration of births across departments for a given name and decade
nb_departements = names.dpt.nunique()   
hhi_dec = birth_numbers_per_decade_per_dpt.assign(sq=birth_numbers_per_decade_per_dpt["share"]**2).groupby(["decade", "preusuel"], as_index=False)["sq"].sum()
hhi_dec["concentration"] = (hhi_dec["sq"] - 1/nb_departements) / (1 - 1/nb_departements)

# popularity per decade and name
totals = (names.groupby(["decade", "preusuel"], as_index=False)["nombre"].sum()
                .rename(columns={"nombre": "total"}))

by_decade = hhi_dec.merge(totals, on=["decade", "preusuel"])
by_decade = by_decade[by_decade["total"] >= 300]

#Add ranking by decade (National popularity rank of the name in the decade)
by_decade["rank"] = (by_decade
    .groupby("decade")["total"]
    .rank(ascending=False)
    .astype(int))

#department with the most births for each decade and name
top_dpt = (birth_numbers_per_decade_per_dpt
    .sort_values("nombre", ascending=False)
    .groupby(["decade", "preusuel"], as_index=False)
    .first()[["decade", "preusuel", "dpt"]]
    .rename(columns={"dpt": "departement"}))


by_decade = by_decade.merge(top_dpt, on=["decade", "preusuel"])

print(by_decade.shape)            
by_decade.head()

(9483, 7)


,decade,preusuel,sq,concentration,total,rank,departement
0,1900,ABEL,0.026285,0.016349,4700,145,62
1,1900,ACHILLE,0.114135,0.105096,1847,216,59
2,1900,ADELAIDE,0.055564,0.045927,1200,266,59
3,1900,ADELINE,0.044626,0.034878,1522,239,59
4,1900,ADOLPHE,0.044385,0.034634,5142,138,59


In [20]:
decades = sorted(by_decade["decade"].unique())

slider = alt.binding_range(min=min(decades), max=max(decades), step=10, name="Decade: ")

pick   = alt.selection_point(fields=["decade"], bind=slider, value=[{"decade": 2010}])

scale=alt.Scale(domain=[0, 1])

zoom = alt.selection_interval(bind="scales", encodings=["x"])
y_midline = alt.Chart(pd.DataFrame({"y": [0.5]})).mark_rule(
    strokeDash=[4, 4], color="grey").encode(y="y")

#x_midline = alt.Chart(pd.DataFrame({"x": [by_decade["rank"].median()]})).mark_rule(
#    strokeDash=[4, 4], color="grey").encode(x="x:Q")



base =alt.Chart(by_decade).add_params(pick, zoom).transform_filter(pick).mark_circle(
    opacity=0.4, color="blue").encode(
    x=alt.X("rank:Q",
        scale=alt.Scale(reverse=True),
        title="Popularity rank in decade (1 = most popular)"),
    y=alt.Y("concentration", scale=alt.Scale(domain=[0, 1]),
            title="Geographic concentration (0 = everywhere · 1 = local)"),
    size=alt.Size("total", legend=None),
    tooltip=["preusuel", "total", "concentration","departement"],
)

#(base + y_midline + x_midline).properties(width=680, height=440,
#             title="Concentration vs popularity, by decade (drag the slider)")

(base + y_midline ).properties(width=680, height=440,
             title="Concentration vs popularity, by decade (drag the slider)")

alt.LayerChart(...)